# Compute Behavioral Measures — Oddball Response

Computes RT, coefficient of variation RT, d-prime, and errors
from aggregated metadata. No average of averages — trials concatenated across runs.

**Input:** `{date}_active_all_metadata.csv`

**Output on NAS (per paper):**
- `{date}_active_{n}p_oddball_rt_{paper}.csv`
- `{date}_active_{n}p_oddball_dprime_{paper}.csv`
- `{date}_active_{n}p_oddball_errors_{paper}.csv`


In [1]:
from datetime import datetime
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from scipy.stats import norm

DATE_STR    = datetime.today().strftime("%Y%m%d")
DATE_SOURCE = "20261016"  # update to match your all_metadata.csv date

if sys.platform == "win32":
    root = Path("L:/EEG_Flow_data/Study_EEG_Flow_Active/derivatives_v2/outputs/measures/behavior")
else:
    root = Path("/Volumes/faculty/EEG_Flow_data/Study_EEG_Flow_Active/derivatives_v2/outputs/measures/behavior")

root.mkdir(parents=True, exist_ok=True)

# Load metadata
fname  = root / f"{DATE_SOURCE}_active_all_metadata.csv"
df_all = pd.read_csv(fname)

print(f"Loaded: {len(df_all):,} rows | {df_all['participant'].nunique()} participants")
print(f"Raw conditions: {sorted(df_all['condition'].unique())}")

# ==============================================================================
# Condition mappings
# ==============================================================================

df_all["condition_flow"] = df_all["condition"].map({
    "low_flow":  "Low Flow",
    "high_flow": "High Flow",
    "oddball_1": "AudioOnly1",
    "oddball_2": "AudioOnly2",
})

df_all["condition_load"] = df_all["condition"].map({
    "oddball_1": "Simple",
    "oddball_2": "Simple",
    "low_flow":  "Complex",
    "high_flow": "Complex",
})

print(f"\nFlow conditions : {sorted(df_all['condition_flow'].unique())}")
print(f"Load conditions : {sorted(df_all['condition_load'].unique())}")


Loaded: 132,910 rows | 49 participants
Raw conditions: ['high_flow', 'low_flow', 'oddball_1', 'oddball_2']

Flow conditions : ['AudioOnly1', 'AudioOnly2', 'High Flow', 'Low Flow']
Load conditions : ['Complex', 'Simple']


In [2]:
# ==============================================================================
# SDT loglinear — Stanislaw & Todorov 1999
# ==============================================================================

def SDT_loglinear(hits, misses, fas, crs):
    """D-prime with log-linear correction."""
    Z        = norm.ppf
    hit_rate = (hits + 0.5) / (hits + misses + 1)
    fa_rate  = (fas  + 0.5) / (fas  + crs   + 1)
    return float(Z(hit_rate) - Z(fa_rate))

print("SDT function ready.")


SDT function ready.


In [3]:
# ==============================================================================
# Compute measures — function per grouping variable
# ==============================================================================

ERROR_TYPES = [
    "FalseAlarms_standard", "FalseAlarms_novel",
    "FalseAlarms_tooquick", "Misses"
]

def compute_rt(df, condition_col):
    """Mean RT and CV for Hits only (response in seconds → ms)."""
    df_hits = df[df["response_type"] == "Hits"].copy()
    df_hits["response_ms"] = df_hits["response"] * 1000
    return (
        df_hits
        .groupby(["participant", condition_col])["response_ms"]
        .agg(rt_mean="mean", rt_std="std", rt_count="count")
        .reset_index()
        .assign(rt_cv=lambda x: (x["rt_std"] / x["rt_mean"]) * 100)
        .rename(columns={condition_col: "condition"})
    )


def compute_dprime_row(df_cond):
    """D-prime for one participant x condition group."""
    hits    = (df_cond["response_type"] == "Hits").sum()
    misses  = (df_cond["response_type"] == "Misses").sum()
    fa_std  = (df_cond["response_type"] == "FalseAlarms_standard").sum()
    fa_nov  = (df_cond["response_type"] == "FalseAlarms_novel").sum()
    fa_tooq = (df_cond["response_type"] == "FalseAlarms_tooquick").sum()
    cr_std  = (df_cond["response_type"] == "CorrectRejections_standard").sum()
    cr_nov  = (df_cond["response_type"] == "CorrectRejections_novel").sum()
    return pd.Series({
        "d_withnovels": SDT_loglinear(hits, misses, fa_std + fa_nov + fa_tooq, cr_std + cr_nov),
        "d_nonovels":   SDT_loglinear(hits, misses, fa_std + fa_tooq, cr_std),
        "n_hits":       hits,
        "n_misses":     misses,
        "n_fa_std":     fa_std,
        "n_fa_nov":     fa_nov,
        "n_fa_tooq":    fa_tooq,
        "n_cr_std":     cr_std,
        "n_cr_nov":     cr_nov,
    })


def compute_dprime(df, condition_col):
    return (
        df
        .groupby(["participant", condition_col])
        .apply(compute_dprime_row)
        .reset_index()
        .rename(columns={condition_col: "condition"})
    )

TOTAL_STIM = {
    "High Flow":  1000, "Low Flow":   1000,
    "AudioOnly1": 360,  "AudioOnly2": 360,
    "Simple":     720,  "Complex":    2000,
}

ERROR_TYPES_NONOVELS = ["FalseAlarms_standard", "FalseAlarms_tooquick", "Misses"]

def compute_errors(df, condition_col):
    df_err = (
        df
        .groupby(["participant", condition_col])["response_type"]
        .apply(lambda x: (x.isin(ERROR_TYPES)).sum())
        .reset_index(name="n_errors")
        .rename(columns={condition_col: "condition"})
    )
    for etype in ERROR_TYPES:
        df_err[etype] = (
            df
            .groupby(["participant", condition_col])["response_type"]
            .apply(lambda x: (x == etype).sum())
            .values
        )
    # Without novels
    df_err["n_errors_nonovels"] = (
        df
        .groupby(["participant", condition_col])["response_type"]
        .apply(lambda x: (x.isin(ERROR_TYPES_NONOVELS)).sum())
        .values
    )
    # Percentages
    df_err["pct_errors"]         = (df_err["n_errors"]         / df_err["condition"].map(TOTAL_STIM)) * 100
    df_err["pct_errors_nonovels"] = (df_err["n_errors_nonovels"] / df_err["condition"].map(TOTAL_STIM)) * 100
    return df_err

print("Functions ready.")


Functions ready.


In [4]:
# ==============================================================================
# Compute for flow and load
# ==============================================================================

df_rt_flow     = compute_rt(df_all, "condition_flow")
df_rt_load     = compute_rt(df_all, "condition_load")

df_dprime_flow = compute_dprime(df_all, "condition_flow")
df_dprime_load = compute_dprime(df_all, "condition_load")

df_errors_flow = compute_errors(df_all, "condition_flow")
df_errors_load = compute_errors(df_all, "condition_load")

print("Flow RT:")
print(df_rt_flow.groupby("condition")["rt_mean"].describe().round(1))
print("\nLoad RT:")
print(df_rt_load.groupby("condition")["rt_mean"].describe().round(1))


Flow RT:
            count   mean   std    min    25%    50%    75%    max
condition                                                        
AudioOnly1   48.0  401.7  69.5  280.3  357.7  388.4  444.7  566.5
AudioOnly2   49.0  398.0  67.0  283.1  352.9  394.1  438.9  586.8
High Flow    49.0  598.2  79.2  423.8  537.8  591.0  663.4  772.2
Low Flow     49.0  598.9  83.8  437.0  529.5  606.1  663.1  816.6

Load RT:
           count   mean   std    min    25%    50%    75%    max
condition                                                       
Complex     49.0  598.4  79.5  432.7  531.5  594.1  662.6  789.9
Simple      49.0  399.2  64.5  281.7  349.4  392.6  450.1  542.6


In [5]:
# ==============================================================================
# Sanity checks
# ==============================================================================

print("=== D-prime withnovels (flow) ===")
print(df_dprime_flow.groupby("condition")["d_withnovels"].describe().round(3))

print("\n=== D-prime withnovels (load) ===")
print(df_dprime_load.groupby("condition")["d_withnovels"].describe().round(3))

print("\n=== Errors (flow) ===")
print(df_errors_flow.groupby("condition")["n_errors"].describe())

print("\n=== Errors (load) ===")
print(df_errors_load.groupby("condition")["n_errors"].describe())

# NaN check
for name, df in [
    ("RT flow",      df_rt_flow),     ("RT load",      df_rt_load),
    ("Dprime flow",  df_dprime_flow),  ("Dprime load",  df_dprime_load),
    ("Errors flow",  df_errors_flow),  ("Errors load",  df_errors_load),
]:
    n_nan = df.isna().sum().sum()
    if n_nan > 0:
        print(f"WARNING — {name}: {n_nan} NaN values")
    else:
        print(f"OK — {name}: no NaN")


=== D-prime withnovels (flow) ===
            count   mean    std    min    25%    50%    75%    max
condition                                                         
AudioOnly1   48.0  4.836  0.454  3.165  4.687  4.993  5.171  5.171
AudioOnly2   49.0  4.715  0.554  2.892  4.413  4.815  5.171  5.171
High Flow    49.0  3.082  0.900  1.423  2.318  3.181  3.654  4.750
Low Flow     49.0  3.030  0.827  1.723  2.477  3.002  3.653  4.836

=== D-prime withnovels (load) ===
           count   mean    std    min    25%    50%    75%    max
condition                                                        
Complex     49.0  3.062  0.858  1.629  2.366  3.076  3.636  4.837
Simple      49.0  5.014  0.595  2.892  4.654  5.209  5.632  5.632

=== Errors (flow) ===
            count       mean        std  min   25%   50%    75%   max
condition                                                            
AudioOnly1   48.0   1.666667   3.680503  0.0   0.0   0.5   1.25  19.0
AudioOnly2   49.0   2.183673   4

In [6]:
# ==============================================================================
# Export to NAS
# ==============================================================================

n = df_all["participant"].nunique()

exports = {
    f"{DATE_STR}_active_oddball_rt_flow.csv":     df_rt_flow,
    f"{DATE_STR}_active_oddball_rt_load.csv":     df_rt_load,
    f"{DATE_STR}_active_oddball_dprime_flow.csv": df_dprime_flow,
    f"{DATE_STR}_active_oddball_dprime_load.csv": df_dprime_load,
    f"{DATE_STR}_active_oddball_errors_flow.csv": df_errors_flow,
    f"{DATE_STR}_active_oddball_errors_load.csv": df_errors_load,
}

for fname, df in exports.items():
    df.to_csv(root / fname, index=False)
    print(f"Saved -> {fname}")


Saved -> 20261016_active_oddball_rt_flow.csv
Saved -> 20261016_active_oddball_rt_load.csv
Saved -> 20261016_active_oddball_dprime_flow.csv
Saved -> 20261016_active_oddball_dprime_load.csv
Saved -> 20261016_active_oddball_errors_flow.csv
Saved -> 20261016_active_oddball_errors_load.csv
